# Iceberg 네임스페이스 관리

로컬 Lakehouse의 `bronze`, `silver`, `gold` 네임스페이스를 생성하고
카탈로그 상태를 안전하게 점검하는 관리용 노트북입니다.

**보여주는 역량:** Spark SQL, Iceberg REST Catalog, 메달리온 계층 관리,
파괴적 DDL에 대한 안전장치

## Goal

- Spark가 `lakehouse` 카탈로그에 연결되는지 확인합니다.
- 메달리온 계층에 대응하는 네임스페이스를 멱등하게 생성합니다.
- 삭제 작업은 명시적으로 허용한 경우에만 실행합니다.

## Setup

Docker Compose 스택이 실행 중이어야 합니다.

```bash
docker compose up -d
```

In [ ]:
from pyspark.sql import SparkSession

CATALOG = "lakehouse"
NAMESPACES = ("bronze", "silver", "gold")

spark = (
    SparkSession.builder
    .appName("iceberg-namespace-admin")
    .getOrCreate()
)
spark.conf.set("spark.sql.session.timeZone", "UTC")

## Steps

### 1. 네임스페이스 생성

In [ ]:
for namespace in NAMESPACES:
    spark.sql(
        f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{namespace}"
    )

spark.sql(f"SHOW NAMESPACES IN {CATALOG}").show(truncate=False)

### 2. 계층별 테이블 확인

In [ ]:
for namespace in NAMESPACES:
    print(f"\n[{CATALOG}.{namespace}]")
    spark.sql(
        f"SHOW TABLES IN {CATALOG}.{namespace}"
    ).show(truncate=False)

### 3. 선택적 삭제

아래 셀은 기본값으로 아무것도 삭제하지 않습니다. 네임스페이스가 비어 있고,
삭제가 필요한 경우에만 `ALLOW_NAMESPACE_DROP = True`로 바꾸세요.

In [ ]:
ALLOW_NAMESPACE_DROP = False

if ALLOW_NAMESPACE_DROP:
    for namespace in reversed(NAMESPACES):
        spark.sql(
            f"DROP NAMESPACE IF EXISTS {CATALOG}.{namespace}"
        )
    print("빈 네임스페이스 삭제를 완료했습니다.")
else:
    print("안전 모드: 삭제 작업을 건너뜁니다.")

## Checks

In [ ]:
namespace_rows = spark.sql(
    f"SHOW NAMESPACES IN {CATALOG}"
).collect()
actual_namespaces = {row.namespace for row in namespace_rows}
missing_namespaces = set(NAMESPACES) - actual_namespaces

assert not missing_namespaces, (
    f"생성되지 않은 네임스페이스: {sorted(missing_namespaces)}"
)
print(f"확인 완료: {', '.join(NAMESPACES)}")

## Next Steps

`pipelines/medallion.ipynb`에서 각 네임스페이스에 Iceberg 테이블을
생성하고 Bronze 데이터를 Silver로 병합합니다.